# MercadoTico 360 · Requisitos del Caso 2 (CouchDB)

Ejecute las celdas **en orden**. Cada sección corresponde a un requisito de la investigación y termina con su evidencia en pantalla. Al final hay un resumen de cumplimiento.

> Requiere haber cargado antes los datos con `Simulacion.ipynb`.

| Requisito | Sección | Evidencia de la demostración |
|---|---|---|
| 1 y 2. Diseño y atributos por categoría | Documentos reales | Visualización de documentos y diseño |
| 3. Índices para 3+ patrones | Índices | Utilidad de cada índice |
| 4. Categoría + precio + atributo | Búsquedas | Pruebas **antes y después** de los índices |
| 5. Pedidos de un cliente y cambio de estado | Pedidos | Pedido anidado, estado actualizado |
| 6. Dos o más agregaciones | MapReduce | Ventas por categoría, ticket promedio, top productos |
| 7. Evolución del esquema | Evolución | Nueva categoría y atributo, validación |

In [ ]:
import json
import os
from getpass import getpass

import requests

try:                                   # Opcional: lee un archivo .env si existe
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass


def leer_config(nombre, pregunta, defecto=None, secreto=False):
    """Toma el valor de una variable de entorno; si no existe, lo pregunta."""
    valor = os.getenv(nombre)
    if valor:
        return valor.strip()
    if secreto:
        return getpass(f"{pregunta}: ").strip()
    sufijo = f" [{defecto}]" if defecto else ""
    return input(f"{pregunta}{sufijo}: ").strip() or defecto


COUCHDB_HOST  = leer_config("COUCHDB_HOST", "Host/IP de CouchDB", "localhost")
COUCHDB_PORT  = leer_config("COUCHDB_PORT", "Puerto de CouchDB")
COUCHDB_USER  = leer_config("COUCHDB_USER", "Usuario de CouchDB", "admin")
COUCHDB_PASS  = leer_config("COUCHDB_PASS", "Contraseña de CouchDB", secreto=True)
DATABASE_NAME = leer_config("DATABASE_NAME", "Nombre de la base de datos", "mercadotico360")

DB_URL = f"http://{COUCHDB_HOST}:{COUCHDB_PORT}/{DATABASE_NAME}"

SESSION = requests.Session()
SESSION.auth = (COUCHDB_USER, COUCHDB_PASS)


## Documentos y atributos variables
**Qué se embebe y qué se referencia**
- `producto`: documento propio; sus `atributos` van **embebidos** y cambian según la categoría.
- `pedido`: las `lineas_detalle` van **embebidas** (con *snapshot* de nombre, categoría y precio); `cliente_id` y `producto_id` son **referencias**.
- `cliente`: documento propio, para que sus datos cambien sin tocar los pedidos.

In [ ]:
def mostrar_productos_variables(categorias=("Ropa", "Tecnologia", "Artesanias")):
    # muestra un producto de cada categoría: los atributos son distintos en cada una.
    for cat in categorias:
        r = SESSION.post(f"{DB_URL}/_find", timeout=60,
                         json={"selector": {"type": "product", "categoria": cat}, "limit": 1}).json()
        if r.get("docs"):
            p = r["docs"][0]
            print(f"{p['_id']} | {p['categoria']} | atributos: {json.dumps(p['atributos'], ensure_ascii=False)}")


def mostrar_pedido_anidado(pedido_id="PED-000001"):
    # muestra un pedido completo con sus líneas de detalle anidadas.
    pedido = SESSION.get(f"{DB_URL}/{pedido_id}", timeout=10).json()
    print(json.dumps(pedido, indent=2, ensure_ascii=False))
    return pedido


print("--- Productos con atributos distintos por categoría ---")
mostrar_productos_variables()
print("\n--- Pedido con estructura anidada ---")
pedido_demo = mostrar_pedido_anidado()
registrar("Diseño y atributos variables", "lineas_detalle" in pedido_demo)

## Pedidos de un cliente y cambio de estado
Para actualizar solo el estado se usa un *update handler* de CouchDB: el cliente envía únicamente `{"estado": ...}` y el servidor aplica el cambio, sin que el código tenga que reenviar el pedido completo.
(CouchDB internamente guarda una nueva revisión del documento.)

In [ ]:
def instalar_update_handler():
    guardar_design_doc({
        "_id": "_design/pedidos",
        "updates": {
            "cambiar_estado": (
                "function (doc, req) {"
                "  if (!doc) { return [null, 'pedido no existe']; }"
                "  doc.estado = JSON.parse(req.body).estado;"
                "  return [doc, 'ok'];"
                "}"
            )
        },
    })


def cambiar_estado_pedido(pedido_id, nuevo_estado):
    url = f"{DB_URL}/_design/pedidos/_update/cambiar_estado/{pedido_id}"
    return SESSION.put(url, json={"estado": nuevo_estado}, timeout=30)


def recuperar_y_actualizar_pedido(cliente_id="CLI-000001", nuevo_estado="completado"):
    # 1. Recuperar los pedidos completos del cliente
    data = buscar(
    {"type": "order", "cliente_id": cliente_id},
    use_index="idx_pedidos_cliente",
    limit=1000,
    titulo=f"Pedidos de {cliente_id}")
    pedidos = data.get("docs", [])
    if not pedidos:
        registrar("Pedidos y actualización", False, "el cliente no tiene pedidos")
        return

    # 2. Actualizar el estado de uno (solo se envía el campo 'estado')
    pedido = next((p for p in pedidos if p["estado"] != nuevo_estado), pedidos[0])   # uno que sí cambie
    anterior = pedido["estado"]
    instalar_update_handler()
    res = cambiar_estado_pedido(pedido["_id"], nuevo_estado)

    # 3. Comprobar el cambio
    actualizado = SESSION.get(f"{DB_URL}/{pedido['_id']}", timeout=10).json()
    print(f"\n{pedido['_id']}: '{anterior}' -> '{actualizado['estado']}'  (HTTP {res.status_code})")
    lineas_iguales = pedido["lineas_detalle"] == actualizado["lineas_detalle"]
    print(
    f"Líneas de detalle conservadas: "
    f"{len(actualizado['lineas_detalle'])} | "
    f"sin cambios: {lineas_iguales}")   
    registrar("Pedidos y actualización de estado", actualizado["estado"] == nuevo_estado and lineas_iguales,
              f"{len(pedidos)} pedidos recuperados")


recuperar_y_actualizar_pedido("CLI-000001")

## Agregaciones con MapReduce
Tres vistas sobre los pedidos (se excluyen los cancelados en ventas y productos vendidos):

| Vista | Qué responde | Reduce |
|---|---|---|
| `ventas_por_categoria` | Ventas totales por categoría (usa el *snapshot* de cada línea) | `_sum` |
| `ticket_por_estado` | Cantidad, suma, mínimo y máximo de pedidos → ticket promedio | `_stats` |
| `productos_mas_vendidos` | Unidades vendidas por producto | `_sum` |

La primera consulta construye la vista y puede tardar un poco.

In [ ]:
def crear_vistas_agregacion():
    guardar_design_doc({
        "_id": "_design/agregaciones",
        "views": {
            "ventas_por_categoria": {
                "map": (
                    "function (doc) { if (doc.type === 'order' && doc.estado !== 'cancelado') {"
                    " doc.lineas_detalle.forEach(function (l) { emit(l.categoria_snapshot, l.subtotal); }); } }"
                ),
                "reduce": "_sum",
            },
            "ticket_por_estado": {
                "map": "function (doc) { if (doc.type === 'order') { emit(doc.estado, doc.monto_total); } }",
                "reduce": "_stats",
            },
            "productos_mas_vendidos": {
                "map": (
                    "function (doc) { if (doc.type === 'order' && doc.estado !== 'cancelado') {"
                    " doc.lineas_detalle.forEach(function (l) { emit(l.producto_id, l.cantidad); }); } }"
                ),
                "reduce": "_sum",
            },
        },
    })


def consultar_vista(nombre):
    url = f"{DB_URL}/_design/agregaciones/_view/{nombre}?group=true"
    respuesta = SESSION.get(url, timeout=600)
    respuesta.raise_for_status()
    return respuesta.json().get("rows", [])


def ejecutar_agregaciones():
    crear_vistas_agregacion()

    print("--- Ventas por categoría ---")
    ventas = sorted(consultar_vista("ventas_por_categoria"), key=lambda r: r["value"], reverse=True)
    for r in ventas:
        print(f"  {str(r['key']):<18} ₡{r['value']:>14,.2f}")

    print("\n--- Ticket promedio por estado de pedido ---")
    for r in consultar_vista("ticket_por_estado"):
        v = r["value"]
        print(f"  {r['key']:<12} pedidos: {v['count']:>6} | promedio: ₡{v['sum'] / v['count']:>9,.2f} "
              f"| mín: ₡{v['min']:,.2f} | máx: ₡{v['max']:,.2f}")

    print("\n--- Top 5 productos más vendidos (unidades) ---")
    top = sorted(consultar_vista("productos_mas_vendidos"), key=lambda r: r["value"], reverse=True)[:5]
    for r in top:
        print(f"  {r['key']}  {r['value']} unidades")

    registrar("Agregaciones MapReduce", bool(ventas and top), "3 vistas consultadas")


ejecutar_agregaciones()

## Evolución del esquema sin migración
Se demuestra con tres acciones sobre la misma base de datos, sin `ALTER TABLE` ni migración:
1. Una **categoría nueva** con atributos nunca vistos.
2. Un **atributo nuevo** en un producto existente.
3. Una **validación** (`validate_doc_update`) que mantiene la consistencia: rechaza documentos inválidos aunque el esquema sea flexible.

In [ ]:
def instalar_validacion():
    guardar_design_doc({
        "_id": "_design/validacion",
        "validate_doc_update": (
            "function (newDoc, oldDoc, userCtx) {"
            "  if (newDoc._deleted || newDoc._id.indexOf('_design/') === 0) { return; }"
            "  if (['product', 'customer', 'order'].indexOf(newDoc.type) === -1) {"
            "    throw ({forbidden: 'tipo de documento inválido'});"
            "  }"
            "  if (newDoc.type === 'product') {"
            "    if (!newDoc.categoria) { throw ({forbidden: 'categoría requerida'}); }"
            "    if (typeof newDoc.precio !== 'number' || newDoc.precio < 0) {"
            "      throw ({forbidden: 'precio inválido'});"
            "    }"
            "  }"
            "  if (newDoc.type === 'order' && "
            "      (!newDoc.lineas_detalle || newDoc.lineas_detalle.length === 0)) {"
            "    throw ({forbidden: 'el pedido necesita líneas'});"
            "  }"
            "}"
        )
    })


def demostrar_evolucion_esquema():
    instalar_validacion()
    print("[1] Validación instalada (consistencia)\n")

    # A) Categoría completamente nueva, con sus propios atributos
    nuevo = {
        "_id": "PROD-999999", "type": "product",
        "nombre": "Drone de Monitoreo Agrícola", "categoria": "Tecnologia_Agricola",
        "precio": 450000, "stock": 10,
        "atributos": {"autonomia_minutos": 45, "camara_multiespectral": True, "alcance_km": 12.5},
    }
    r = SESSION.put(f"{DB_URL}/PROD-999999", json=nuevo, timeout=30)
    estado = {201: "creada", 202: "creada", 409: "ya existía"}.get(r.status_code, f"error {r.status_code}")
    print(f"[2] Categoría nueva 'Tecnologia_Agricola': {estado}")
    hallados = buscar({"type": "product", "categoria": "Tecnologia_Agricola"},
                      use_index="idx_product_category", titulo="    consulta de la categoría nueva")

    # B) Atributo nuevo en un producto existente
    ropa = SESSION.post(f"{DB_URL}/_find", timeout=60,
                        json={"selector": {"type": "product", "categoria": "Ropa"}, "limit": 1}).json()["docs"][0]
    ropa["atributos"]["certificacion_ecologica"] = True
    r2 = SESSION.put(f"{DB_URL}/{ropa['_id']}", json=ropa, timeout=30)
    print(f"[3] Atributo nuevo 'certificacion_ecologica' en {ropa['_id']}: HTTP {r2.status_code}")

    # C) Consistencia: un producto inválido es rechazado por la base de datos
    invalido = {"type": "product", "categoria": "Ropa", "nombre": "Prueba", "precio": -5}
    r3 = SESSION.post(DB_URL, json=invalido, timeout=30)
    print(f"[4] Producto con precio negativo: HTTP {r3.status_code} -> {r3.json().get('reason')}")

    registrar("Evolución del esquema",
              r.status_code in (201, 202, 409) and bool(hallados.get("docs")) and r3.status_code == 403 and r2.status_code in (201, 202),
              "categoría nueva, atributo nuevo y validación")


demostrar_evolucion_esquema()

## Resumen de pruebas

In [ ]:
print("RESUMEN DE PRUEBAS - MERCADOTICO 360")
for prueba, (ok, detalle) in RESULTADOS.items():
    estado = "Correcto" if ok else "Revisar"
    print(f"{estado}: {prueba} - {detalle}")
print(
    f"\nPruebas correctas: "
    f"{sum(ok for ok, _ in RESULTADOS.values())}/{len(RESULTADOS)}"
)